# 01 – Exploratory Data Analysis (EDA) SIDER

**Tuần 2 – ADR Project 1 HUST 2026-1**

Notebook này gọi lại module `adr` thay vì giữ logic pipeline riêng.  
**Không chứa kết quả giả.** Chạy notebook sau khi có `data/raw/sider.csv`.

Trước khi chạy:
```powershell
python -m adr.data.download --config configs/base.yaml
python -m adr.data.prepare --config configs/base.yaml
```

In [ ]:
from pathlib import Path
import sys

# Đảm bảo import được adr package (chạy sau pip install -e .)
ROOT = Path().resolve().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from adr.config import load_config, resolve_paths
from adr.data.validate import validate_raw, print_report

print('Import OK')

In [ ]:
# Load config
cfg = resolve_paths(load_config(ROOT / 'configs' / 'base.yaml'))
raw_dir = cfg['paths']['data_raw']
processed_dir = cfg['paths']['data_processed']
raw_csv = raw_dir / 'sider.csv'

print(f'Raw CSV: {raw_csv}')
print(f'Tồn tại: {raw_csv.exists()}')

In [ ]:
# ─── Kiểm tra file tồn tại trước khi đọc ───
if not raw_csv.exists():
    print('Chưa có raw data. Chạy: python -m adr.data.download --config configs/base.yaml')
else:
    df = pd.read_csv(raw_csv)
    print(f'Shape: {df.shape}')
    print(f'Số thuốc (hàng): {len(df)}')
    print(f'Cột: {list(df.columns[:5])}...')
    df.head(3)

In [ ]:
# ─── Validate schema ───
if raw_csv.exists():
    report = validate_raw(raw_csv)
    print_report(report)
    print(f'\nLabel names (27 theo thứ tự cột nguồn):')
    for i, name in enumerate(report.label_names):
        print(f'  {i+1:2d}. {name}')

In [ ]:
# ─── Phân phối positive rate và missing rate per nhãn ───
if raw_csv.exists():
    label_cols = report.label_names
    pos_rates = []
    missing_rates = []
    for col in label_cols:
        n_obs = df[col].notna().sum()
        n_pos = (df[col] == 1).sum()
        pos_rates.append(n_pos / n_obs if n_obs > 0 else 0)
        missing_rates.append(df[col].isna().sum() / len(df))

    fig, axes = plt.subplots(2, 1, figsize=(16, 10))

    # Positive rate
    axes[0].bar(range(len(label_cols)), pos_rates, color='steelblue', alpha=0.8)
    axes[0].set_xticks(range(len(label_cols)))
    axes[0].set_xticklabels(label_cols, rotation=90, fontsize=7)
    axes[0].set_ylabel('Positive rate (trong observed)')
    axes[0].set_title('Positive Rate per nhãn ADR (chỉ tính trên nhãn quan sát được)')
    axes[0].axhline(0.5, color='red', linestyle='--', label='0.5')
    axes[0].legend()

    # Missing rate
    axes[1].bar(range(len(label_cols)), missing_rates, color='salmon', alpha=0.8)
    axes[1].set_xticks(range(len(label_cols)))
    axes[1].set_xticklabels(label_cols, rotation=90, fontsize=7)
    axes[1].set_ylabel('Missing rate')
    axes[1].set_title('Missing rate per nhãn ADR')

    plt.tight_layout()
    plt.savefig(ROOT / 'reports' / 'label_distribution.png', dpi=100)
    plt.show()

In [ ]:
# ─── Phân phối độ dài SMILES ───
if raw_csv.exists():
    smiles_lengths = df['smiles'].str.len()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(smiles_lengths, bins=50, color='teal', alpha=0.8, edgecolor='white')
    ax.set_xlabel('Độ dài SMILES (ký tự)')
    ax.set_ylabel('Số phân tử')
    ax.set_title(f'Phân phối độ dài SMILES (n={len(df)})')
    ax.axvline(smiles_lengths.median(), color='red', linestyle='--', label=f'Median={smiles_lengths.median():.0f}')
    ax.axvline(smiles_lengths.max(), color='orange', linestyle='--', label=f'Max={smiles_lengths.max()}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(ROOT / 'reports' / 'smiles_length_dist.png', dpi=100)
    plt.show()
    print(f'Median: {smiles_lengths.median():.0f}, Max: {smiles_lengths.max()}, Min: {smiles_lengths.min()}')

In [ ]:
# ─── Số label dương trung bình mỗi phân tử ───
if raw_csv.exists():
    label_sum = (df[label_cols] == 1).sum(axis=1)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(label_sum, bins=20, color='purple', alpha=0.8, edgecolor='white')
    ax.set_xlabel('Số nhãn dương mỗi thuốc')
    ax.set_ylabel('Số phân tử')
    ax.set_title('Phân phối số nhãn dương (multi-label nature)')
    plt.tight_layout()
    plt.show()
    print(f'Trung bình: {label_sum.mean():.2f} nhãn/thuốc')
    print(f'Median: {label_sum.median():.0f}, Max: {label_sum.max()}')

In [ ]:
# ─── Processed data (sau khi chạy prepare) ───
processed_csv = processed_dir / 'processed.csv'
if processed_csv.exists():
    df_proc = pd.read_csv(processed_csv)
    print(f'Processed: {len(df_proc)} thuốc (sau dedup & canonicalize)')
    print(f'drug_id sample: {df_proc["drug_id"].iloc[0][:20]}...')
    
    label_names_path = processed_dir / 'label_names.json'
    if label_names_path.exists():
        with label_names_path.open() as f:
            label_names = json.load(f)
        print(f'\n27 label names (theo thứ tự cột nguồn):')
        for i, n in enumerate(label_names):
            print(f'  {i+1:2d}. {n}')
else:
    print('Chưa có processed.csv. Chạy: python -m adr.data.prepare --config configs/base.yaml')